# Predator–Mangsa dengan Imigrasi

**ID proyek:** `O005-LEGA-V101-PRJ08`  
**Status:** titik awal pedagogis yang ditulis secara independen.

Notebook ini menggunakan data sintetis/terbuka saja. Notebook ini **bukan** kode atau data dari makalah yang dikutip dalam bab sumber dan **bukan** klaim reproduksi hasil penelitian mana pun.


## Pertanyaan pemodelan

Bagaimana aliran mangsa dari luar sistem menggeser kesetimbangan dan lintasan predator–mangsa?

Tujuan kerja: tetapkan sistem, jalankan eksperimen deterministik, periksa invarian, visualisasikan perilaku, lalu kritik kecukupan model.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

SEED = 2026082208
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)


## Struktur dan asumsi

Interaksi mengikuti Lotka–Volterra dengan laju imigrasi mangsa konstan; parameter dan lingkungan tetap; kedua populasi kontinu.

Semua skala dan parameter di notebook ini bersifat ilustratif. Ubah satu asumsi pada satu waktu dan catat dampaknya pada keluaran serta invarian.


In [ ]:
r, attack, conversion, mortality = 0.80, 0.70, 0.50, 0.30
t_eval = np.linspace(0.0, 100.0, 801)

def equilibrium(immigration):
    prey = mortality / (conversion * attack)
    predator = (r * prey + immigration) / (attack * prey)
    return np.array([prey, predator])

def predator_prey_run(immigration):
    def rhs(t, y):
        prey, predator = y
        return [r * prey - attack * prey * predator + immigration, conversion * attack * prey * predator - mortality * predator]
    eq = equilibrium(immigration)
    initial = eq * np.array([0.72, 1.25])
    return solve_ivp(rhs, (0.0, 100.0), initial, t_eval=t_eval, rtol=1e-9, atol=1e-11), eq, rhs

immigration_runs = {m: predator_prey_run(m) for m in (0.0, 0.12)}


## Pemeriksaan numerik

Pemeriksaan berikut sengaja berada di dalam notebook: eksekusi berhenti bila suatu invarian dasar gagal. Ini bukan bukti bahwa model benar; ini hanya bukti bahwa implementasi memenuhi kontrak numerik terbatasnya.


In [ ]:
for immigration, (run, eq, rhs) in immigration_runs.items():
    assert run.success and np.isfinite(run.y).all() and np.min(run.y) > 0.0
    np.testing.assert_allclose(rhs(0.0, eq), [0.0, 0.0], atol=1e-12)
assert immigration_runs[0.12][1][1] > immigration_runs[0.0][1][1]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for immigration, (run, eq, _) in immigration_runs.items():
    axes[0].plot(t_eval, run.y[0], label=f"mangsa, m={immigration}")
    axes[0].plot(t_eval, run.y[1], linestyle="--", label=f"predator, m={immigration}")
    axes[1].plot(run.y[0], run.y[1], label=f"m={immigration}")
    axes[1].scatter(*eq, s=25)
axes[0].set(xlabel="waktu", ylabel="populasi", title="Deret waktu")
axes[1].set(xlabel="mangsa", ylabel="predator", title="Bidang fase")
axes[0].legend(fontsize=7)
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()
plt.close(fig)


## Validasi, identifikasi, dan keterbatasan

Keterbatasan awal: Tidak ada daya dukung, struktur umur, musim, stokastisitas demografis, atau umpan balik pada imigrasi.

Jawab sebelum menafsirkan gambar:

1. Besaran apa yang benar-benar dapat diamati, dan bagaimana galat pengukurannya dimodelkan?
2. Parameter mana yang dapat diidentifikasi dari keluaran tersebut? Tunjukkan dengan profil galat, pemisahan latih/uji, atau eksperimen sensitivitas.
3. Invarian atau pola kualitatif apa yang harus tetap benar ketika ukuran langkah, benih acak, atau resolusi diubah?
4. Temukan satu skenario kegagalan model dan jelaskan data tambahan yang diperlukan untuk membedakannya dari model alternatif.


## Daftar periksa reproduksibilitas

- [ ] Gunakan CPython dan versi paket tepat seperti `requirements.lock`.
- [ ] Jalankan ulang dari kernel kosong tanpa jaringan.
- [ ] Pertahankan nilai `SEED` (benih acak), lalu ulangi dengan sedikitnya lima benih acak lain dan laporkan variasinya.
- [ ] Catat setiap perubahan parameter, persamaan, toleransi, serta pembagian data.
- [ ] Pastikan semua uji lulus dan jelaskan mengapa tiap uji relevan.
- [ ] Simpan hasil turunan di luar notebook sumber; notebook distribusi harus tetap tanpa keluaran tersimpan.
- [ ] Bedakan hasil simulasi, data sintetis, dan klaim empiris secara eksplisit.
